# PDS 37 with VLTI/GRAVITY: joint binary fit to all epochs

All GRAVITY exposures are fitted together with one binary model. Combining epochs fills in the uv plane and
improves the constraints, **but it assumes the companion did not move between observations**. Compare the
result with the epoch-by-epoch fits in `03_gravity_binary_fits_by_epoch.ipynb` before trusting it.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmoired

sys.path.insert(0, str(Path.cwd().parent))  # make the local `pds37` package importable
from pds37 import config

plt.rcParams["figure.dpi"] = 80
from pds37.binary import (binary_grid_search, with_brg_lines, image_window,
                          fit_table, FLUX_RATIO_PRIOR, BRG_LINE_PRIORS)

ALL_FILES = [str(p) for p in config.gravity_files()]

## 1. All data at full spectral resolution

In [ ]:
oi = pmoired.OI(ALL_FILES, insname=config.GRAVITY_INSNAME, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.K_BAND]})
oi.show(allInOne=True)

In [ ]:
oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.BRG_WINDOW]})
oi.show(allInOne=True)

## 2. Binary of two uniform disks

Grid search over ±30 mas in 2 mas steps on the binned (×20) K-band data, with the primary flux fixed to 1.

In [ ]:
oi = pmoired.OI(ALL_FILES, insname=config.GRAVITY_INSNAME, binning=20, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI"], "wl ranges": [config.K_BAND]})

start = binary_grid_search(oi, max_sep=30, step=2,
                           model={"A,ud": 0.2, "A,f": 1, "B,ud": 0.2, "B,f": 0.2},
                           do_not_fit=["A,f", "A,ud", "B,ud"])
oi.showGrid(interpolate=True, logV=1, cmap="gist_stern")

In [ ]:
oi.doFit(start, doNotFit=["A,f"], verbose=1)
ud_fit = {"best": dict(oi.bestfit["best"]), "uncer": dict(oi.bestfit["uncer"]), "chi2": oi.bestfit["chi2"]}
oi.show(**image_window(ud_fit["best"]))

## 3. Alternative: binary of two Gaussians

Same geometry, but each component is a Gaussian (size given as FWHM) instead of a sharp-edged disk, which better
describes a star surrounded by diffuse circumstellar emission. Starting values come from the uniform-disk fit.

In [ ]:
# fresh object so no priors from the uniform-disk grid search carry over
oi = pmoired.OI(ALL_FILES, insname=config.GRAVITY_INSNAME, binning=20, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI"], "wl ranges": [config.K_BAND]})

ud = ud_fit["best"]
gauss_start = {"A,profile": "gaussian", "A,f": 1, "A,fwhm": 1.3,
               "B,profile": "gaussian", "B,f": ud["B,f"], "B,fwhm": 10,
               "B,x": ud["B,x"], "B,y": ud["B,y"]}
oi.doFit(gauss_start, doNotFit=["A,f"], prior=FLUX_RATIO_PRIOR, verbose=1)
gauss_fit = {"best": dict(oi.bestfit["best"]), "uncer": dict(oi.bestfit["uncer"]), "chi2": oi.bestfit["chi2"]}
oi.show(**image_window(gauss_fit["best"]))

pd.DataFrame({"uniform disks": [ud_fit["chi2"]], "Gaussians": [gauss_fit["chi2"]]},
             index=["reduced χ²"]).round(2)

### Bootstrap uncertainties for the Gaussian model

In [ ]:
oi.bootstrapFit(Nfits=500, verbose=0)
oi.showBootstrap()
gauss_boot = {"best": dict(oi.boot["best"]), "uncer": dict(oi.boot["uncer"])}

## 4. Brackett-γ emission

Full spectral resolution around Brγ, starting from the uniform-disk binary and adding one Lorentzian emission
line per component.

In [ ]:
oi = pmoired.OI(ALL_FILES, insname=config.GRAVITY_INSNAME, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI", "NFLUX", "DPHI"],
             "wl ranges": [config.BRG_WINDOW],
             "NFLUX order": 2})
oi.doFit(with_brg_lines(ud_fit["best"]), prior=BRG_LINE_PRIORS, doNotFit=["A,f"], verbose=1)
oi.show(**image_window(oi.bestfit["best"]), allInOne=True)

## Summary of the joint fits

In [ ]:
fit_table({"uniform-disk binary": ud_fit,
           "Gaussian binary (bootstrap)": gauss_boot,
           "uniform-disk binary + Brγ": oi.bestfit},
          ["A,ud", "A,fwhm", "B,ud", "B,fwhm", "B,f", "B,x", "B,y",
           "A,line_brg_f", "B,line_brg_f"])